# Target-indication pair tables

One row per target-indication pair, with the maximum clinical phase reached and the genetic
support propagated through the disease ontology. Oncology indications (descendants of
`MONDO_0045024`) are removed. Two support columns are carried: `score_all` from every
L2G-prioritised credible set, and `score_pav` from those containing a protein-altering
variant. Methods "Clinical trials success modelling".

Writes `ti_pairs_chembl`, `l2g_indirect_assoc_all`, `l2g_indirect_assoc_pav`, and, if the
Pharmaprojects table is available, `ti_pairs_pharmaprojects`.

In [1]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

from manuscript_methods import paper
from manuscript_methods.enrichment import or_rs, support_mask

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 01:21:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sl = StudyLocus.from_parquet(session, paper.release("credible_set"))
si = StudyIndex.from_parquet(session, paper.release("study"))
disease_index = session.spark.read.parquet(paper.release("disease") + "/disease.parquet")
chembl_evidence = session.spark.read.parquet(paper.release("evidence") + "/sourceId=chembl")

l2g = session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
genes = session.spark.read.parquet(paper.derived("gene_table"))
print("L2G rows:", l2g.count(), "genes with pleiotropy:", genes.count())

L2G rows: 70400 genes with pleiotropy: 8285


## ChEMBL target-indication pairs, oncology removed

In [3]:
efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index, efo_ids=["MONDO_0045024"]
)
chembl = chemblDrugEnrichment.process_chembl_evidence(chembl_evidence, efo_to_remove).cache()
n_chembl = chembl.count()
print("ChEMBL T-I pairs:", n_chembl)
chembl.groupBy("maxClinicalPhase").count().orderBy("maxClinicalPhase").show()

ChEMBL T-I pairs: 37377


+----------------+-----+
|maxClinicalPhase|count|
+----------------+-----+
|             1.0| 6163|
|             2.0|14410|
|             3.0|12240|
|             4.0| 4564|
+----------------+-----+



## Genetic support propagated through the ontology

In [4]:
def indirect_assoc(table):
    """Propagate the L2G scores of one credible-set subset through the disease ontology."""
    evidence = chemblDrugEnrichment.to_disease_target_evidence(
        table_with_score=table.drop("diseaseIds"),
        score_column="score",
        datasource_id="l2g",
        study_locus=sl,
        study_index=si,
        min_score=0.1,
    )
    return chemblDrugEnrichment.evidence_to_indirect_assosiations(
        evidence, disease_index, use_max=True, efo_to_remove=efo_to_remove
    ).cache()


def propagate_features(table):
    """Propagate score, effect size, MAF and PAV status to every ontology ancestor.

    Same propagation as `indirect_assoc`, but carrying the variant-level features the Figure 5b
    strata and the Figure 5c regression need. Only credible sets scoring at least 0.1 contribute.
    """
    exploded = (
        table.filter(f.col("score") >= 0.1)
        .withColumn("diseaseId", f.explode("diseaseIds"))
        .withColumnRenamed("geneId", "targetId")
        .filter(~f.col("diseaseId").isin(efo_to_remove))
    )
    ancestors = disease_index.select(
        f.col("id").alias("diseaseId"), f.explode("ancestors").alias("ancestorDiseaseId")
    ).union(disease_index.select(f.col("id").alias("diseaseId"), f.col("id").alias("ancestorDiseaseId")))
    return (
        exploded.join(ancestors, "diseaseId", "inner")
        .groupBy("targetId", "ancestorDiseaseId")
        .agg(
            f.max("score").alias("indirect_assoc_score"),
            f.max("absBeta").alias("max_beta"),
            f.min("maf").alias("min_maf"),
            f.max("VEP").alias("max_vep"),
        )
        .withColumnRenamed("ancestorDiseaseId", "diseaseId")
        .cache()
    )


# Each stratum is propagated on its own subset of credible sets, so a pair counts as
# rare-supported only when a rare credible set carries the support. Filtering the propagated
# table afterwards is not the same thing.
SUBSETS = {
    "pav": f.col("VEP") == 1,
    "rare": f.col("maf") < 0.01,
    "common": f.col("maf") >= 0.01,
    "large_effect": f.col("absBeta") > 0.5,
    "small_effect": f.col("absBeta") <= 0.5,
}

assoc_all = indirect_assoc(l2g)
assoc_pav = indirect_assoc(l2g.filter(SUBSETS["pav"]))
strata = {name: indirect_assoc(l2g.filter(condition)) for name, condition in SUBSETS.items()}
features = propagate_features(l2g)
for name, table in strata.items():
    print(f"  support from {name}: {table.count()} target-disease pairs")
print("target-disease pairs with support (all):", assoc_all.count(), "(PAV):", assoc_pav.count())
print("propagated feature rows:", features.count())

for name, table in [("all", assoc_all), ("pav", assoc_pav), ("features", features)]:
    assert table.count() == table.select("targetId", "diseaseId").distinct().count(), name
assoc_all.write.mode("overwrite").parquet(paper.derived("l2g_indirect_assoc_all"))
assoc_pav.write.mode("overwrite").parquet(paper.derived("l2g_indirect_assoc_pav"))
features.write.mode("overwrite").parquet(paper.derived("l2g_indirect_assoc_features"))

26/08/19 01:21:59 WARN CacheManager: Asked to cache already cached data.


  support from pav: 22509 target-disease pairs


  support from rare: 5183 target-disease pairs


  support from common: 149301 target-disease pairs


  support from large_effect: 13550 target-disease pairs


  support from small_effect: 144215 target-disease pairs


target-disease pairs with support (all): 151704 (PAV): 22509


propagated feature rows: 151704


## Master table

In [5]:
gene_features = genes.select(
    f.col("geneId").alias("targetId"), "uniqueTherapeuticAreas", "uniqueDiseases", "approvedSymbol"
)

master = chembl.join(
    assoc_all.withColumnRenamed("indirect_assoc_score", "score_all"), ["targetId", "diseaseId"], "left"
)
for name, table in strata.items():
    master = master.join(
        table.withColumnRenamed("indirect_assoc_score", f"score_{name}"), ["targetId", "diseaseId"], "left"
    )
master = (
    master.join(features.drop("indirect_assoc_score"), ["targetId", "diseaseId"], "left")
    .join(gene_features, "targetId", "left")
    .toPandas()
)
assert len(master) == n_chembl, "joins changed the number of ChEMBL pairs"

master["in_gps"] = master["uniqueTherapeuticAreas"].notna()
master["approved"] = (master["maxClinicalPhase"] >= 4).astype(int)
master.to_parquet(paper.derived("ti_pairs_chembl"), index=False)

print("pairs:", len(master))
print("with any genetic support:", int(master["score_all"].notna().sum()))
print("with PAV genetic support:", int(master["score_pav"].notna().sum()))
print("approved:", int(master["approved"].sum()))

# The PAV flag propagated with the features must agree with the separate PAV propagation.
print("max_vep == 1 rows:", int((master["max_vep"] == 1).sum()))

pairs: 37377
with any genetic support: 742
with PAV genetic support: 161
approved: 4564
max_vep == 1 rows: 161


## Control: the two headline enrichment estimates

In [6]:
checks = pd.DataFrame(
    [
        {"definition": "all GWAS", **or_rs(support_mask(master), master["approved"])},
        {"definition": "PAV + 2-5 TA", **or_rs(support_mask(master, pav=True, ta_min=2, ta_max=5), master["approved"])},
    ]
).set_index("definition")
print(checks[["odds_ratio", "relative_success", "yes_evid-high_clinphase", "p_value"]].to_string())
print()
print("manuscript: all GWAS OR 3.62 with 242 approved; PAV + 2-5 TA OR 10.3, RS 4.8, 51 approved")

              odds_ratio  relative_success  yes_evid-high_clinphase       p_value
definition                                                                       
all GWAS        3.618578          2.764540                      242  3.534571e-49
PAV + 2-5 TA   10.288962          4.843708                       51  8.072448e-25

manuscript: all GWAS OR 3.62 with 242 approved; PAV + 2-5 TA OR 10.3, RS 4.8, 51 approved


## Pharmaprojects

Used by Supplementary Results 11 as an independently curated comparison. The processed table
comes from `chapters/_legacy/05-other-drug-indication-data`; it is a licensed resource and is
skipped when absent.

In [7]:
from pathlib import Path

pharmaprojects = Path(paper.baseline("minikel_etal_processed_data_v2.csv"))
if not pharmaprojects.exists():
    print("Pharmaprojects table absent, skipping (GAPS.md)")
else:
    columns = [
        "targetId",
        "diseaseId",
        "meshId",
        "approvedSymbol",
        "ti_uid",
        "ccatnum",
        "maxClinicalPhase",
        "outcome",
        "geneticSupport_old",
        "genetic_insight",
        "target_status",
        "year_launch",
        "orphan",
    ]
    pp = pd.read_csv(pharmaprojects, low_memory=False)[columns].copy()
    pp_master = (
        pp.merge(
            assoc_all.toPandas().rename(columns={"indirect_assoc_score": "score_all"}),
            on=["targetId", "diseaseId"],
            how="left",
        )
        .merge(
            assoc_pav.toPandas().rename(columns={"indirect_assoc_score": "score_pav"}),
            on=["targetId", "diseaseId"],
            how="left",
        )
        .merge(gene_features.toPandas().drop(columns=["approvedSymbol"]), on="targetId", how="left")
    )
    assert len(pp_master) == len(pp), "joins changed the number of Pharmaprojects pairs"
    pp_master["in_gps"] = pp_master["uniqueTherapeuticAreas"].notna()
    pp_master["approved"] = pp_master["outcome"].astype(int)
    pp_master.to_parquet(paper.derived("ti_pairs_pharmaprojects"), index=False)
    print("Pharmaprojects pairs:", len(pp_master), "launched:", int(pp_master["approved"].sum()))
    # Their own genetic-support flag against their own outcome, as a provenance check.
    own = or_rs(pp_master["geneticSupport_old"].astype(bool), pp_master["approved"])
    print(
        "their flag: OR",
        round(own["odds_ratio"], 3),
        "log-odds ~ 0.843 in Minikel et al.:",
        round(np.log(own["odds_ratio"]), 4),
    )

Pharmaprojects pairs: 7390 launched: 913
their flag: OR 2.323 log-odds ~ 0.843 in Minikel et al.: 0.8431
